# Qwen3-Reranker-0.6B: vLLM inference на 10 000 товарных пар

Notebook проверяет zero-shot inference исходной модели
`Qwen/Qwen3-Reranker-0.6B` на стратифицированной выборке ручной
разметки. Обучения здесь нет. Цели эксперимента:

- проверить совместимость оригинального checkpoint с `vLLM.score`;
- измерить model-load time, throughput и длины входов на двух T4;
- получить overall и macro average precision до fine-tuning;
- сохранить predictions и hard examples в `/kaggle/working`.

Выборка содержит только 10 000 пар и подключается из отдельного
приватного Kaggle Dataset. Notebook приватный; Kaggle-токен и полные
parquet-файлы в него не включаются.

In [ ]:
import importlib.metadata
import json
import os
import subprocess
import sys
import time
from pathlib import Path

WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
VLLM_VERSION = "0.14.0"

try:
    installed_vllm = importlib.metadata.version("vllm")
except importlib.metadata.PackageNotFoundError:
    installed_vllm = None

if installed_vllm != VLLM_VERSION:
    print(f"Installing vLLM {VLLM_VERSION}; found {installed_vllm!r}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}"],
        check=True,
    )
else:
    print(f"vLLM {installed_vllm} is already installed")

print(subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True).stdout)

In [ ]:
import json
import re

import pandas as pd

sample_candidates = list(
    Path("/kaggle/input").glob("**/qwen3_inference_sample_10k.parquet")
)
if len(sample_candidates) != 1:
    raise RuntimeError(
        "Expected one attached qwen3_inference_sample_10k.parquet, "
        f"found {sample_candidates}"
    )
pairs = pd.read_parquet(sample_candidates[0])
assert len(pairs) == 10_000
assert set(pairs["target"].unique()) == {0, 1}
print(
    f"Loaded {len(pairs):,} pairs from {sample_candidates[0]}; "
    f"positive rate={pairs.target.mean():.2%}"
)
display(pd.crosstab(pairs["category"], pairs["target"], margins=True))

## Сериализация карточек

Ключи не удаляются по глобальной частоте. Сначала идут потенциальные
идентификаторы и variant-поля, затем все остальные атрибуты в
детерминированном порядке. Очень длинное отдельное значение и общий
текст ограничиваются защитными лимитами; точное token-truncation до 448
токенов на карточку выполняется внутри inference worker.

In [ ]:
PRIORITY_PATTERNS = [
    r"артикул|партномер|oem|штрих|ean|gtin|код товара|код производителя|sku",
    r"бренд|производитель|модель",
    r"тип|вид ",
    r"размер|цвет|объем|объём|вес|количество|комплектац",
]
MAX_VALUE_CHARS = 500
MAX_ITEM_CHARS = 8_000


def clean(value):
    return " ".join(str(value).replace("ё", "е").split())


def key_priority(key):
    normalized = clean(key).lower()
    for priority, pattern in enumerate(PRIORITY_PATTERNS):
        if re.search(pattern, normalized):
            return priority, normalized
    return len(PRIORITY_PATTERNS), normalized


def serialize_item(category, name, raw_attributes):
    attributes = json.loads(raw_attributes)
    lines = [f"Категория: {clean(category)}", f"Название: {clean(name)}", "Атрибуты:"]
    for key, value in sorted(attributes.items(), key=lambda item: key_priority(item[0])):
        value = clean(value)
        if not value:
            continue
        lines.append(f"{clean(key)}: {value[:MAX_VALUE_CHARS]}")
    return "\n".join(lines)[:MAX_ITEM_CHARS]


pairs["text1"] = [
    serialize_item(category, name, attributes)
    for category, name, attributes in zip(pairs.category, pairs.name1, pairs.attributes1)
]
pairs["text2"] = [
    serialize_item(category, name, attributes)
    for category, name, attributes in zip(pairs.category, pairs.name2, pairs.attributes2)
]
display(pairs[["text1", "text2", "target"]].head(2))
print(pairs[["text1", "text2"]].map(len).describe(percentiles=[.5, .9, .95, .99]))

input_columns = ["pair_index", "id1", "id2", "target", "category", "text1", "text2"]
input_path = WORKING_DIR / "reranker_input.csv.gz"
pairs[input_columns].to_csv(input_path, index=False, compression="gzip")
print(f"Wrote {input_path} ({input_path.stat().st_size / 2**20:.2f} MiB)")

## vLLM inference

Worker запускается отдельным Python-процессом. Это изолирует vLLM и
его CUDA/PyTorch зависимости от уже запущенного Jupyter kernel. Исходный
Qwen checkpoint подключается к pooling runner через официальные
`hf_overrides`; `tensor_parallel_size=2` задействует обе T4. Backend
внимания явно задан как `TRITON_ATTN`: он совместим с T4 и не требует
проблемной FlashInfer JIT-сборки в Kaggle image.

In [ ]:
WORKER_PATH = WORKING_DIR / "run_qwen3_vllm_worker.py"
WORKER_PATH.write_text('\nfrom __future__ import annotations\n\nimport json\nimport os\nimport subprocess\nimport time\nfrom pathlib import Path\n\n# FlashInfer tries to JIT-link against an unversioned libcuda.so that is absent\n# from the Kaggle T4 image. vLLM\'s Triton backend supports all attention types\n# and CUDA compute capabilities without that FlashInfer linker path.\nos.environ.setdefault("VLLM_ATTENTION_BACKEND", "TRITON_ATTN")\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom transformers import AutoTokenizer\nfrom vllm import LLM\n\n\nWORKING_DIR = Path("/kaggle/working")\nINPUT_PATH = WORKING_DIR / "reranker_input.csv.gz"\nPREDICTION_PATH = WORKING_DIR / "qwen3_reranker_predictions.parquet"\nRUNTIME_PATH = WORKING_DIR / "qwen3_reranker_runtime.json"\n\nMODEL_NAME = "Qwen/Qwen3-Reranker-0.6B"\nMAX_ITEM_TOKENS = 448\nMAX_MODEL_LEN = 1024\nEXPECTED_GPU_COUNT = 2\n\nCHAT_TEMPLATE = r\'\'\'<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n<Instruct>: Determine whether the Query and Document describe the same marketplace product. Treat wording and schema differences as irrelevant, but distinguish genuinely different models or material product configurations.\n<Query>: {{ messages | selectattr("role", "eq", "query") | map(attribute="content") | first }}\n<Document>: {{ messages | selectattr("role", "eq", "document") | map(attribute="content") | first }}<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\'\'\'\n\n\ndef batched_truncate(tokenizer, texts: list[str], max_tokens: int, batch_size: int = 128):\n    truncated_texts: list[str] = []\n    original_lengths: list[int] = []\n    for start in range(0, len(texts), batch_size):\n        batch = texts[start : start + batch_size]\n        encoded = tokenizer(batch, add_special_tokens=False, truncation=False)["input_ids"]\n        original_lengths.extend(len(token_ids) for token_ids in encoded)\n        truncated_texts.extend(\n            tokenizer.batch_decode(\n                [token_ids[:max_tokens] for token_ids in encoded],\n                skip_special_tokens=True,\n            )\n        )\n    return truncated_texts, np.asarray(original_lengths, dtype=np.int32)\n\n\ndef main() -> None:\n    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")\n    os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")\n    gpu_count = torch.cuda.device_count()\n    if gpu_count != EXPECTED_GPU_COUNT:\n        raise RuntimeError(f"Expected {EXPECTED_GPU_COUNT} GPUs, got {gpu_count}")\n\n    frame = pd.read_csv(INPUT_PATH, compression="gzip")\n    queries = frame["text1"].astype(str).tolist()\n    documents = frame["text2"].astype(str).tolist()\n\n    tokenizer_start = time.perf_counter()\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n    queries, query_lengths = batched_truncate(tokenizer, queries, MAX_ITEM_TOKENS)\n    documents, document_lengths = batched_truncate(tokenizer, documents, MAX_ITEM_TOKENS)\n    tokenization_seconds = time.perf_counter() - tokenizer_start\n\n    load_start = time.perf_counter()\n    llm = LLM(\n        model=MODEL_NAME,\n        runner="pooling",\n        hf_overrides={\n            "architectures": ["Qwen3ForSequenceClassification"],\n            "classifier_from_token": ["no", "yes"],\n            "is_original_qwen3_reranker": True,\n        },\n        tensor_parallel_size=gpu_count,\n        distributed_executor_backend="mp",\n        dtype="float16",\n        max_model_len=MAX_MODEL_LEN,\n        max_num_seqs=128,\n        gpu_memory_utilization=0.85,\n        enforce_eager=True,\n        seed=42,\n    )\n    model_load_seconds = time.perf_counter() - load_start\n\n    inference_start = time.perf_counter()\n    outputs = llm.score(queries, documents, chat_template=CHAT_TEMPLATE)\n    inference_seconds = time.perf_counter() - inference_start\n    scores = np.asarray([output.outputs.score for output in outputs], dtype=np.float32)\n    if len(scores) != len(frame) or not np.isfinite(scores).all():\n        raise RuntimeError("vLLM returned missing or non-finite scores")\n\n    predictions = frame[["pair_index", "id1", "id2", "target", "category"]].copy()\n    predictions["predict"] = scores\n    predictions.to_parquet(PREDICTION_PATH, index=False)\n\n    gpu_info = subprocess.run(\n        [\n            "nvidia-smi",\n            "--query-gpu=index,name,memory.total,memory.used",\n            "--format=csv,noheader,nounits",\n        ],\n        check=False,\n        capture_output=True,\n        text=True,\n    ).stdout.strip().splitlines()\n    all_lengths = np.concatenate([query_lengths, document_lengths])\n    runtime = {\n        "model": MODEL_NAME,\n        "attention_backend": os.environ["VLLM_ATTENTION_BACKEND"],\n        "pairs": int(len(frame)),\n        "gpu_count": gpu_count,\n        "gpu_info_after_inference": gpu_info,\n        "max_item_tokens": MAX_ITEM_TOKENS,\n        "max_model_len": MAX_MODEL_LEN,\n        "tokenization_seconds": tokenization_seconds,\n        "model_load_seconds": model_load_seconds,\n        "inference_seconds": inference_seconds,\n        "pairs_per_second": len(frame) / inference_seconds,\n        "token_length_p50": float(np.quantile(all_lengths, 0.50)),\n        "token_length_p95": float(np.quantile(all_lengths, 0.95)),\n        "token_length_p99": float(np.quantile(all_lengths, 0.99)),\n        "token_length_max": int(all_lengths.max()),\n        "item_truncation_rate": float((all_lengths > MAX_ITEM_TOKENS).mean()),\n    }\n    RUNTIME_PATH.write_text(\n        json.dumps(runtime, ensure_ascii=False, indent=2), encoding="utf-8"\n    )\n    print(json.dumps(runtime, ensure_ascii=False, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', encoding="utf-8")
print(f"Worker written to {WORKER_PATH}")

worker_start = time.perf_counter()
subprocess.run([sys.executable, str(WORKER_PATH)], check=True)
worker_wall_seconds = time.perf_counter() - worker_start
print(f"Worker wall time: {worker_wall_seconds:.1f} seconds")

## Метрики и диагностические примеры

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import average_precision_score

predictions = pd.read_parquet(WORKING_DIR / "qwen3_reranker_predictions.parquet")
runtime = json.loads((WORKING_DIR / "qwen3_reranker_runtime.json").read_text())
assert len(predictions) == 10_000
assert np.isfinite(predictions["predict"]).all()

per_category_ap = predictions.groupby("category").apply(
    lambda part: average_precision_score(part["target"], part["predict"]),
    include_groups=False,
).sort_values()
macro_ap = float(per_category_ap.mean())
overall_ap = float(average_precision_score(predictions["target"], predictions["predict"]))

metrics = {
    **runtime,
    "overall_average_precision": overall_ap,
    "macro_average_precision": macro_ap,
    "score_min": float(predictions["predict"].min()),
    "score_median": float(predictions["predict"].median()),
    "score_max": float(predictions["predict"].max()),
    "per_category_average_precision": per_category_ap.to_dict(),
}
(WORKING_DIR / "qwen3_reranker_metrics.json").write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2), encoding="utf-8"
)
display(pd.Series({
    "macro AP": macro_ap,
    "overall AP": overall_ap,
    "pairs/second": runtime["pairs_per_second"],
    "inference seconds": runtime["inference_seconds"],
    "model load seconds": runtime["model_load_seconds"],
    "item truncation rate": runtime["item_truncation_rate"],
}).to_frame("value"))
display(per_category_ap.to_frame("average_precision"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for target, label, color in [(0, "не дубль", "#64748b"), (1, "дубль", "#f97316")]:
    values = predictions.loc[predictions.target == target, "predict"]
    axes[0].hist(values, bins=50, density=True, histtype="step", linewidth=2,
                 label=label, color=color)
axes[0].set(title="Zero-shot score distribution", xlabel="Qwen3 reranker score", ylabel="density")
axes[0].legend()
per_category_ap.plot.barh(ax=axes[1], color="#0f766e")
axes[1].set(title="Average precision по категориям", xlabel="AP", ylabel="")
fig.tight_layout()
figure_path = WORKING_DIR / "qwen3_reranker_diagnostics.png"
fig.savefig(figure_path, dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
review = predictions.merge(
    pairs[["pair_index", "name1", "name2"]], on="pair_index", how="left", validate="one_to_one"
)
hard_negatives = review[review.target == 0].nlargest(20, "predict").assign(slice="high_scored_negative")
hard_positives = review[review.target == 1].nsmallest(20, "predict").assign(slice="low_scored_positive")
examples = pd.concat([hard_negatives, hard_positives], ignore_index=True)
examples.to_csv(WORKING_DIR / "qwen3_reranker_hard_examples.csv", index=False)
display(examples[["slice", "category", "target", "predict", "name1", "name2"]])

completion = {
    "status": "complete",
    "pairs": len(predictions),
    "macro_average_precision": macro_ap,
    "overall_average_precision": overall_ap,
}
(WORKING_DIR / "notebook_completed.json").write_text(
    json.dumps(completion, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(json.dumps(completion, ensure_ascii=False, indent=2))

## Интерпретация

Это zero-shot проверка retrieval-reranker на задаче product identity,
поэтому качество не является baseline обученного решения. Главные
результаты notebook — работоспособность `vLLM.score`, фактическая
скорость на 2×T4, доля truncation и распределение scores. Следующий
честный эксперимент — fine-tuning на component-disjoint train folds и
повторный inference на зафиксированном holdout.